# Urban Flow Analytics Datathon 2026
## 01 — Exploratory / Data Quality / Data Cleaning

This notebook is designed for the **complete 12-month Urban Flow Analytics Taxi Dataset**, not only the February sample.

### Objectives

- Load and validate all 12 monthly Taxi CSV files.
- Load and validate the Zone Dataset.
- Establish the raw-data baseline.
- Check schema, data types, missing values, duplicates, and categorical values.
- Calculate trip duration and speed.
- Detect the five required anomaly categories.
- Quantify every anomaly by count and percentage of raw rows.
- Document the decision to drop, filter, impute, or retain.
- Investigate anomaly overlap and monthly variation.
- Create a clean audit-ready dataset.
- Validate location IDs against the Zone Dataset.
- Enrich pickup/drop-off locations.
- Save outputs for later prediction and analytics.

> **Important:** Actual counts and percentages are calculated from all 12 months. Do not use numbers from the February sample in the final report.

## Challenge Requirement

The challenge requires identification and handling of:

1. Negative fares
2. Distance equal to 0 with a non-zero fare
3. Rider/passenger count equal to 0
4. Drop-off before pickup
5. Unrealistic speeds based on distance and time

For every anomaly, report:

- affected row count;
- percentage of total raw rows;
- treatment: drop, filter, impute, or retain;
- justification for the treatment.

The raw data must remain available so the cleaning process is auditable.

## Cleaning Philosophy

This notebook follows four principles:

1. Never overwrite the raw data.
2. Create anomaly flags before deleting rows.
3. Do not impute values that cannot be reliably reconstructed.
4. Distinguish globally invalid rows from task-specific filters.

For example, a zero-distance trip may be unsuitable for speed analysis without necessarily being invalid for fare analysis.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import json
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries loaded successfully.")

## 2. Project Paths

Expected structure:

```text
Urban_Flow_Analytics/
├── data/
│   ├── raw/
│   │   ├── taxi/
│   │   │   ├── month_01.csv
│   │   │   ├── ...
│   │   │   └── month_12.csv
│   │   └── zones/
│   │       └── Urban_Flow_Analytics_Zone_Dataset.csv
│   ├── interim/
│   └── processed/
├── reports/
└── notebooks/
```

Change the paths in the next cell if your project folders are different.

In [ ]:
# ============================================================
# 2. Project paths
# ============================================================

PROJECT_ROOT = Path("..").resolve()

RAW_TAXI_DIR = PROJECT_ROOT / "data" / "raw" / "taxi"
RAW_ZONE_DIR = PROJECT_ROOT / "data" / "raw" / "zones"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

for folder in [INTERIM_DIR, PROCESSED_DIR, REPORT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Taxi folder:", RAW_TAXI_DIR)
print("Zone folder:", RAW_ZONE_DIR)

## 3. Find the 12 Monthly Taxi Files

The final run must contain exactly 12 monthly Taxi CSV files.

The February dataset can be used while developing/debugging the notebook, but the final execution must use all 12.

In [ ]:
# ============================================================
# 3. Discover Taxi files
# ============================================================

taxi_files = sorted(RAW_TAXI_DIR.glob("*.csv"))

print("Taxi CSV files found:", len(taxi_files))

for i, file in enumerate(taxi_files, 1):
    print(f"{i:02d}. {file.name}")

if len(taxi_files) != 12:
    raise ValueError(
        f"Expected 12 monthly Taxi CSV files, found {len(taxi_files)}. "
        "Check data/raw/taxi/ before continuing."
    )

## 4. Inspect Headers Before Combining

Inspect a few rows and the column names from each month.

This catches schema differences before millions of rows are combined.

In [ ]:
# ============================================================
# 4. Inspect each monthly file
# ============================================================

for file in taxi_files:
    sample = pd.read_csv(file, nrows=3, low_memory=False)

    print("=" * 90)
    print(file.name)
    print("Rows inspected:", len(sample))
    print("Column count:", len(sample.columns))
    print("Columns:", sample.columns.tolist())
    display(sample)

## 5. Validate Column Consistency

All 12 monthly files should have compatible column names.

If a month has missing/extra columns, investigate it before concatenation rather than silently accepting the mismatch.

In [ ]:
# ============================================================
# 5. Compare monthly schemas
# ============================================================

column_sets = {
    file.name: set(pd.read_csv(file, nrows=0).columns)
    for file in taxi_files
}

reference_file = taxi_files[0].name
reference_columns = column_sets[reference_file]

schema_issues = []

for filename, columns in column_sets.items():
    missing = sorted(reference_columns - columns)
    extra = sorted(columns - reference_columns)

    if missing or extra:
        schema_issues.append({
            "file": filename,
            "missing_columns": missing,
            "extra_columns": extra
        })

if schema_issues:
    display(pd.DataFrame(schema_issues))
else:
    print("PASS: all 12 monthly files have the same column set.")

## 6. Load All 12 Months

A `source_file` field is added only for auditability and monthly quality analysis.

In [ ]:
# ============================================================
# 6. Load and combine all 12 months
# ============================================================

monthly_frames = []

for file in taxi_files:
    print("Loading:", file.name)

    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = file.name

    monthly_frames.append(temp)

taxi_raw = pd.concat(monthly_frames, ignore_index=True)

del monthly_frames
gc.collect()

print("Combined shape:", taxi_raw.shape)

## 7. Raw Baseline

The raw row count is the denominator for the required anomaly percentages.

\[
Impact\% =
\frac{Affected\ Rows}{Total\ Raw\ Rows}\times100
\]

This baseline must not change after cleaning.

In [ ]:
# ============================================================
# 7. Establish raw baseline
# ============================================================

initial_rows = len(taxi_raw)

print(f"Raw rows: {initial_rows:,}")
print(f"Raw columns: {taxi_raw.shape[1]:,}")

monthly_raw_counts = (
    taxi_raw.groupby("source_file")
    .size()
    .rename("rows")
    .to_frame()
)

display(monthly_raw_counts)

## 8. Inspect Data Types and Basic Statistics

In [ ]:
# ============================================================
# 8. Basic inspection
# ============================================================

display(taxi_raw.head())
display(taxi_raw.dtypes.to_frame("dtype"))

numeric_columns = taxi_raw.select_dtypes(include=np.number).columns.tolist()

display(
    taxi_raw[numeric_columns]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.999])
    .T
)

## 9. Convert Timestamp Fields

Invalid timestamp values are converted to `NaT` so they can be counted and investigated.

In [ ]:
# ============================================================
# 9. Timestamp conversion
# ============================================================

taxi_raw["pickup_timestamp"] = pd.to_datetime(
    taxi_raw["pickup_timestamp"],
    errors="coerce"
)

taxi_raw["dropoff_timestamp"] = pd.to_datetime(
    taxi_raw["dropoff_timestamp"],
    errors="coerce"
)

timestamp_report = pd.DataFrame({
    "field": ["pickup_timestamp", "dropoff_timestamp"],
    "missing_or_invalid": [
        taxi_raw["pickup_timestamp"].isna().sum(),
        taxi_raw["dropoff_timestamp"].isna().sum()
    ]
})

timestamp_report["percentage_of_raw_rows"] = (
    timestamp_report["missing_or_invalid"] / initial_rows * 100
)

display(timestamp_report)

## 9b. Column Type Standardisation

After timestamp conversion, explicitly cast remaining columns to their
correct dtypes.

- String/object columns that represent a finite set of categories are
  cast to `category` — this reduces memory and enables validation.
- Numeric columns accidentally loaded as `object` (mixed types from CSV)
  are coerced with `pd.to_numeric(..., errors='coerce')`.

Invalid numeric entries become `NaN` and are captured in the
missing-value report that follows.

In [ ]:
# ============================================================
# 9b. Type standardisation
# ============================================================

# --- Categorical columns ----------------------------------------
# Adjust this list to match the actual column names in your CSV.
CATEGORY_COLS = [
    col for col in [
        "vendor_id",
        "payment_type",
        "fare_settlement_method",
        "rate_code",
        "trip_type",
        "store_and_fwd_flag",
    ]
    if col in taxi_raw.columns
]

for col in CATEGORY_COLS:
    taxi_raw[col] = taxi_raw[col].astype("category")

# --- Numeric columns that may have been read as object ----------
EXPECTED_NUMERIC_COLS = [
    col for col in [
        "rider_count",
        "passenger_count",
        "distance_miles",
        "base_fare",
        "charge_total",
        "driver_tip_payment",
        "toll_total",
        "origin_loc_id",
        "dest_loc_id",
    ]
    if col in taxi_raw.columns
    and taxi_raw[col].dtype == object
]

for col in EXPECTED_NUMERIC_COLS:
    taxi_raw[col] = pd.to_numeric(taxi_raw[col], errors="coerce")

print(f"Category columns cast   : {CATEGORY_COLS}")
print(f"Numeric columns coerced : {EXPECTED_NUMERIC_COLS}")
print()
display(taxi_raw.dtypes.to_frame("dtype_after_standardisation"))


## 9c. Categorical Value Validation

Inspect the distinct values of every categorical column.

Goals:

1. Detect unexpected codes, typos, or mixed-case variants
   (e.g. `"CASH"` vs `"cash"` vs `"Cash "`).
2. Detect columns that have suspiciously high cardinality
   (possible free-text leaking into a categorical field).
3. Flag rows whose category value is not in an accepted set,
   if the valid set is known from the data dictionary.

In [ ]:
# ============================================================
# 9c. Categorical value validation
# ============================================================

cat_validation_rows = []

for col in CATEGORY_COLS:
    vc = taxi_raw[col].value_counts(dropna=False)
    n_unique = taxi_raw[col].nunique(dropna=False)
    n_null = taxi_raw[col].isna().sum()

    cat_validation_rows.append({
        "column": col,
        "n_unique_values": n_unique,
        "null_count": n_null,
        "null_pct": round(n_null / initial_rows * 100, 4),
        "top_values": ", ".join(
            str(v) for v in vc.index[:5].tolist()
        )
    })

    print(f"\n--- {col} (unique={n_unique}, nulls={n_null}) ---")
    display(vc.rename("count").to_frame())

cat_validation_summary = pd.DataFrame(cat_validation_rows)
print("\n=== Categorical Validation Summary ===")
display(cat_validation_summary)

# ---------------------------------------------------------------
# Normalise string categories: strip whitespace + lower-case.
# Re-run value_counts after normalisation to confirm.
# ---------------------------------------------------------------
for col in CATEGORY_COLS:
    # Remove leading/trailing whitespace and unify case
    taxi_raw[col] = (
        taxi_raw[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", None)
        .astype("category")
    )

print("\nCategory values normalised (strip + lower).")


## 10. Missing-Value Analysis

Missing values are analyzed separately from the five required anomaly categories.

Do not automatically impute every missing value. The later predictive task may require different treatment for different variables.

In [ ]:
# ============================================================
# 10. Missing-value report
# ============================================================

missing_report = pd.DataFrame({
    "missing_count": taxi_raw.isna().sum(),
    "missing_percentage": taxi_raw.isna().mean() * 100
}).sort_values("missing_percentage", ascending=False)

display(missing_report)

## 11. Duplicate Analysis

Exact duplicates are reported but not automatically removed.

A duplicate may be a true duplicate, a repeated operational record, or an ingestion problem, so it requires investigation.

In [ ]:
# ============================================================
# 11. Duplicate analysis
# ============================================================

duplicate_count = taxi_raw.duplicated().sum()
duplicate_percentage = duplicate_count / initial_rows * 100

print(f"Duplicate rows excluding first occurrence: {duplicate_count:,}")
print(f"Percentage of raw rows: {duplicate_percentage:.4f}%")

if duplicate_count:
    display(taxi_raw[taxi_raw.duplicated(keep=False)].head(20))

# 12. Required Anomaly Detection

The next sections create flags only. No rows are deleted until the anomaly counts, percentages, and decisions have been documented.

## 12.1 Negative Fares

**Detection rule**

```text
base_fare < 0
```

**Decision:** exclude from normal fare-prediction data.

**Reason:** a negative value is not a missing value and cannot safely be replaced with an estimated fare. The original record remains in the audit data.

Before finalizing, inspect settlement-method values because the dataset contains special settlement categories such as dispute and voided trip.

In [ ]:
# ============================================================
# 12.1 Negative fares
# ============================================================

taxi_raw["flag_negative_fare"] = taxi_raw["base_fare"] < 0

count = taxi_raw["flag_negative_fare"].sum()
percentage = count / initial_rows * 100

print(f"Affected rows: {count:,}")
print(f"Percentage of raw rows: {percentage:.4f}%")

display(
    taxi_raw.loc[
        taxi_raw["flag_negative_fare"],
        [
            "pickup_timestamp",
            "dropoff_timestamp",
            "distance_miles",
            "base_fare",
            "charge_total",
            "fare_settlement_method"
        ]
    ].head(20)
)

if "fare_settlement_method" in taxi_raw.columns:
    display(
        taxi_raw.loc[
            taxi_raw["flag_negative_fare"],
            "fare_settlement_method"
        ].value_counts(dropna=False).rename("count").to_frame()
    )

## 12.2 Zero Distance With Non-Zero Fare

**Detection rule**

```text
distance_miles == 0 AND base_fare != 0
```

**Decision:** retain globally after investigation, but filter from movement/speed analysis.

**Reason:** zero measured distance does not necessarily prove that a fare record is invalid. It does, however, prevent meaningful speed calculation.

In [ ]:
# ============================================================
# 12.2 Zero distance + non-zero fare
# ============================================================

taxi_raw["flag_zero_distance_nonzero_fare"] = (
    (taxi_raw["distance_miles"] == 0) &
    (taxi_raw["base_fare"] != 0)
)

count = taxi_raw["flag_zero_distance_nonzero_fare"].sum()
percentage = count / initial_rows * 100

print(f"Affected rows: {count:,}")
print(f"Percentage of raw rows: {percentage:.4f}%")

display(
    taxi_raw.loc[
        taxi_raw["flag_zero_distance_nonzero_fare"],
        [
            "pickup_timestamp",
            "dropoff_timestamp",
            "distance_miles",
            "base_fare",
            "surcharge_misc",
            "transit_tax",
            "charge_total"
        ]
    ].head(20)
)

## 12.3 Zero Rider Count

**Detection rule**

```text
rider_count == 0
```

**Decision:** filter from analyses that require a valid passenger count.

**Reason:** the true rider count cannot be reliably reconstructed from the available fields, so unsupported imputation is avoided.

In [ ]:
# ============================================================
# 12.3 Zero rider count
# ============================================================

taxi_raw["flag_zero_riders"] = taxi_raw["rider_count"] == 0

count = taxi_raw["flag_zero_riders"].sum()
percentage = count / initial_rows * 100

print(f"Affected rows: {count:,}")
print(f"Percentage of raw rows: {percentage:.4f}%")

display(
    taxi_raw.loc[
        taxi_raw["flag_zero_riders"],
        [
            "rider_count",
            "distance_miles",
            "base_fare",
            "charge_total",
            "pickup_timestamp",
            "dropoff_timestamp"
        ]
    ].head(20)
)

## 12.4 Drop-Off Before Pickup

**Detection rule**

```text
dropoff_timestamp < pickup_timestamp
```

**Decision:** drop from the cleaned modelling dataset.

**Reason:** the temporal ordering is impossible, resulting in a negative trip duration. The correct timestamp cannot be inferred reliably, so imputation would fabricate information.

In [ ]:
# ============================================================
# 12.4 Drop-off before pickup
# ============================================================

taxi_raw["flag_dropoff_before_pickup"] = (
    taxi_raw["dropoff_timestamp"] <
    taxi_raw["pickup_timestamp"]
)

count = taxi_raw["flag_dropoff_before_pickup"].sum()
percentage = count / initial_rows * 100

print(f"Affected rows: {count:,}")
print(f"Percentage of raw rows: {percentage:.4f}%")

display(
    taxi_raw.loc[
        taxi_raw["flag_dropoff_before_pickup"],
        [
            "pickup_timestamp",
            "dropoff_timestamp",
            "distance_miles",
            "base_fare"
        ]
    ].head(20)
)

## 12.5 Trip Duration

Trip duration is calculated from the timestamps.

\[
Duration = Dropoff\ Timestamp - Pickup\ Timestamp
\]

In [ ]:
# ============================================================
# 12.5 Trip duration
# ============================================================

taxi_raw["trip_duration_seconds"] = (
    taxi_raw["dropoff_timestamp"] -
    taxi_raw["pickup_timestamp"]
).dt.total_seconds()

taxi_raw["trip_duration_minutes"] = (
    taxi_raw["trip_duration_seconds"] / 60
)

taxi_raw["trip_duration_hours"] = (
    taxi_raw["trip_duration_seconds"] / 3600
)

display(
    taxi_raw[
        [
            "pickup_timestamp",
            "dropoff_timestamp",
            "trip_duration_minutes"
        ]
    ].head()
)

print(
    "Negative duration rows:",
    (taxi_raw["trip_duration_seconds"] < 0).sum()
)

## 12.6 Unrealistic Speed

Speed is calculated as:

\[
Speed_{mph} = \frac{distance\_miles}{trip\_duration\_hours}
\]

Only positive-duration trips are used.

An extreme statistical value is not automatically an invalid record. The threshold must be justified using the empirical distribution and operational context.

In [ ]:
# ============================================================
# 12.6 Calculate speed
# ============================================================

taxi_raw["speed_mph"] = np.nan

valid_speed = (
    (taxi_raw["trip_duration_hours"] > 0) &
    (taxi_raw["distance_miles"] >= 0)
)

taxi_raw.loc[valid_speed, "speed_mph"] = (
    taxi_raw.loc[valid_speed, "distance_miles"] /
    taxi_raw.loc[valid_speed, "trip_duration_hours"]
)

display(
    taxi_raw["speed_mph"]
    .describe(
        percentiles=[
            0.50, 0.90, 0.95, 0.99, 0.995, 0.999
        ]
    )
    .to_frame()
)

In [ ]:
# ============================================================
# 12.7 Speed distribution
# ============================================================

plt.figure(figsize=(10, 5))

plt.hist(
    taxi_raw["speed_mph"].dropna(),
    bins=100
)

plt.xlim(0, 100)
plt.xlabel("Speed (mph)")
plt.ylabel("Number of trips")
plt.title("Trip Speed Distribution")

plt.tight_layout()
plt.show()

## 12.8 Select the Speed Threshold

The value below is a configurable project decision.

Before final submission, the team must validate it by inspecting:

1. high speed percentiles;
2. extreme records;
3. trip duration;
4. distance;
5. operational plausibility.

Do not claim that the threshold is an official competition threshold unless the organizers provide one.

In [ ]:
# ============================================================
# 12.8 Speed threshold
# ============================================================

# Project decision. Validate and document this threshold before final submission.
SPEED_THRESHOLD_MPH = 60.0

print(f"Documented threshold: {SPEED_THRESHOLD_MPH:.1f} mph")

In [ ]:
# ============================================================
# 12.9 Flag unrealistic speed
# ============================================================

taxi_raw["flag_unrealistic_speed"] = (
    taxi_raw["speed_mph"] > SPEED_THRESHOLD_MPH
)

count = taxi_raw["flag_unrealistic_speed"].sum()
percentage = count / initial_rows * 100

print(f"Affected rows: {count:,}")
print(f"Percentage of raw rows: {percentage:.4f}%")

display(
    taxi_raw.loc[
        taxi_raw["flag_unrealistic_speed"],
        [
            "source_file",
            "pickup_timestamp",
            "dropoff_timestamp",
            "distance_miles",
            "trip_duration_minutes",
            "speed_mph"
        ]
    ]
    .sort_values("speed_mph", ascending=False)
    .head(30)
)

# 13. Required Anomaly Summary

This is the main table for the technical report.

All percentages use the complete raw 12-month row count as the denominator.

In [ ]:
# ============================================================
# 13. Anomaly summary
# ============================================================

FLAG_COLUMNS = [
    "flag_negative_fare",
    "flag_zero_distance_nonzero_fare",
    "flag_zero_riders",
    "flag_dropoff_before_pickup",
    "flag_unrealistic_speed"
]

anomaly_summary = pd.DataFrame({
    "Anomaly": [
        "Negative base fare",
        "Zero distance + non-zero fare",
        "Zero rider count",
        "Drop-off before pickup",
        "Unrealistic speed"
    ],
    "Affected Rows": [
        taxi_raw[c].sum()
        for c in FLAG_COLUMNS
    ],
    "Decision": [
        "Exclude from normal fare-prediction dataset",
        "Retain globally; filter from movement/speed analysis",
        "Filter from passenger-count-dependent analysis",
        "Drop from cleaned modelling dataset",
        "Filter from movement/travel-time analysis"
    ],
    "Impute": [
        "No",
        "No",
        "No",
        "No",
        "No"
    ]
})

anomaly_summary["Percentage of Raw Rows"] = (
    anomaly_summary["Affected Rows"] /
    initial_rows * 100
)

display(anomaly_summary)

# 14. Anomaly Overlap

A trip can contain multiple anomaly types.

Therefore, the sum of anomaly counts is not necessarily the number of unique affected rows.

In [ ]:
# ============================================================
# 14. Anomaly overlap
# ============================================================

taxi_raw["any_required_anomaly"] = (
    taxi_raw[FLAG_COLUMNS].any(axis=1)
)

taxi_raw["number_of_required_anomalies"] = (
    taxi_raw[FLAG_COLUMNS].sum(axis=1)
)

unique_affected = taxi_raw["any_required_anomaly"].sum()

print(f"Unique affected rows: {unique_affected:,}")
print(
    f"Percentage of raw rows: "
    f"{unique_affected / initial_rows * 100:.4f}%"
)

display(
    taxi_raw["number_of_required_anomalies"]
    .value_counts()
    .sort_index()
    .rename("rows")
    .to_frame()
)

# 15. Data Quality Decision Matrix

The following matrix records the treatment rationale in a form suitable for the report.

In [ ]:
# ============================================================
# 15. Decision matrix
# ============================================================

decision_matrix = pd.DataFrame({
    "Anomaly": [
        "Negative base fare",
        "Zero distance + non-zero fare",
        "Zero rider count",
        "Drop-off before pickup",
        "Unrealistic speed"
    ],
    "Detection": [
        "base_fare < 0",
        "distance_miles == 0 AND base_fare != 0",
        "rider_count == 0",
        "dropoff_timestamp < pickup_timestamp",
        f"speed_mph > {SPEED_THRESHOLD_MPH}"
    ],
    "Treatment": [
        "Drop/filter for fare prediction",
        "Task-specific filter",
        "Filter",
        "Drop",
        "Task-specific filter"
    ],
    "Justification": [
        "Negative fare is not a missing value and should not be replaced by an invented target.",
        "Zero distance does not prove the fare is invalid, but it cannot support meaningful movement/speed analysis.",
        "The actual rider count cannot be reliably reconstructed from the available fields.",
        "The temporal sequence is impossible and the correct timestamp cannot be inferred.",
        "The derived speed is operationally implausible; imputation would hide the underlying inconsistency."
    ]
})

display(decision_matrix)

# 16. Monthly Data Quality Analysis

Aggregate results can hide month-specific problems.

Calculate anomaly counts and anomaly percentages separately for every monthly file.

In [ ]:
# ============================================================
# 16. Monthly anomaly analysis
# ============================================================

monthly_quality = (
    taxi_raw
    .groupby("source_file")
    .agg(
        total_rows=("source_file", "size"),
        negative_fares=("flag_negative_fare", "sum"),
        zero_distance_nonzero_fare=("flag_zero_distance_nonzero_fare", "sum"),
        zero_riders=("flag_zero_riders", "sum"),
        dropoff_before_pickup=("flag_dropoff_before_pickup", "sum"),
        unrealistic_speed=("flag_unrealistic_speed", "sum")
    )
)

monthly_quality_percent = monthly_quality.copy()

for col in [
    "negative_fares",
    "zero_distance_nonzero_fare",
    "zero_riders",
    "dropoff_before_pickup",
    "unrealistic_speed"
]:
    monthly_quality_percent[col] = (
        monthly_quality[col] /
        monthly_quality["total_rows"] *
        100
    )

display(monthly_quality)
display(monthly_quality_percent)

In [ ]:
# ============================================================
# 16.1 Monthly anomaly-rate visualization
# ============================================================

monthly_quality_percent[
    [
        "negative_fares",
        "zero_distance_nonzero_fare",
        "zero_riders",
        "dropoff_before_pickup",
        "unrealistic_speed"
    ]
].plot(
    kind="bar",
    figsize=(14, 6)
)

plt.xlabel("Source month/file")
plt.ylabel("Percentage of monthly rows")
plt.title("Monthly Data Quality Anomaly Rates")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 17. Inspect Extreme Records

Before applying the final speed rule, inspect the most extreme observations.

In [ ]:
# ============================================================
# 17. Extreme observations
# ============================================================

print("Highest speeds")
display(
    taxi_raw[
        [
            "source_file",
            "distance_miles",
            "trip_duration_minutes",
            "speed_mph"
        ]
    ]
    .sort_values("speed_mph", ascending=False)
    .head(20)
)

print("Largest distances")
display(
    taxi_raw[
        [
            "source_file",
            "distance_miles",
            "trip_duration_minutes",
            "speed_mph"
        ]
    ]
    .sort_values("distance_miles", ascending=False)
    .head(20)
)

print("Largest base fares")
display(
    taxi_raw[
        [
            "source_file",
            "distance_miles",
            "trip_duration_minutes",
            "base_fare",
            "charge_total"
        ]
    ]
    .sort_values("base_fare", ascending=False)
    .head(20)
)

# 18. Create the Quality-Audit Dataset

No required anomaly is deleted here.

The audit dataset contains the raw observations plus derived values and anomaly flags.

In [ ]:
# ============================================================
# 18. Save audit dataset
# ============================================================

taxi_quality = taxi_raw.copy()

quality_audit_path = INTERIM_DIR / "taxi_quality_audit_12month.parquet"

taxi_quality.to_parquet(
    quality_audit_path,
    index=False
)

print("Saved:", quality_audit_path)

# 19. Apply Main Cleaning Decisions

The main cleaned dataset applies the conservative rules documented above:

- drop negative fares from the normal fare-prediction dataset;
- filter zero-rider records from the main passenger-aware dataset;
- drop impossible drop-off-before-pickup records;
- filter unrealistic-speed records from the main travel-time/movement dataset.

Zero-distance/non-zero-fare records are retained globally and handled as a task-specific filter because the anomaly does not prove the fare record itself is invalid.

In [ ]:
# ============================================================
# 19. Build cleaned dataset
# ============================================================

taxi_clean = taxi_quality.copy()

taxi_clean = taxi_clean[
    ~taxi_clean["flag_dropoff_before_pickup"]
].copy()

taxi_clean = taxi_clean[
    ~taxi_clean["flag_negative_fare"]
].copy()

taxi_clean = taxi_clean[
    ~taxi_clean["flag_zero_riders"]
].copy()

taxi_clean = taxi_clean[
    ~taxi_clean["flag_unrealistic_speed"]
].copy()

print(f"Raw rows: {initial_rows:,}")
print(f"Clean rows: {len(taxi_clean):,}")
print(f"Rows removed: {initial_rows - len(taxi_clean):,}")
print(
    f"Removed percentage: "
    f"{(initial_rows - len(taxi_clean)) / initial_rows * 100:.4f}%"
)

## 19.1 Movement/Speeed Analysis Dataset

Zero-distance records are excluded here because speed requires positive distance.

This is a task-specific filter rather than a global deletion.

In [ ]:
# ============================================================
# 19.1 Movement-analysis filter
# ============================================================

taxi_movement = taxi_clean[
    (taxi_clean["distance_miles"] > 0) &
    (taxi_clean["trip_duration_seconds"] > 0)
].copy()

print(f"Clean main rows: {len(taxi_clean):,}")
print(f"Movement-analysis rows: {len(taxi_movement):,}")

# 20. Before vs After Cleaning

In [ ]:
# ============================================================
# 20. Before/after summary
# ============================================================

before_after = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Negative fares",
        "Zero distance + non-zero fare",
        "Zero riders",
        "Drop-off before pickup",
        "Unrealistic speed",
        "Rows removed by main cleaning"
    ],
    "Raw": [
        initial_rows,
        taxi_quality["flag_negative_fare"].sum(),
        taxi_quality["flag_zero_distance_nonzero_fare"].sum(),
        taxi_quality["flag_zero_riders"].sum(),
        taxi_quality["flag_dropoff_before_pickup"].sum(),
        taxi_quality["flag_unrealistic_speed"].sum(),
        initial_rows - len(taxi_clean)
    ],
    "Clean": [
        len(taxi_clean),
        (taxi_clean["base_fare"] < 0).sum(),
        (
            (taxi_clean["distance_miles"] == 0) &
            (taxi_clean["base_fare"] != 0)
        ).sum(),
        (taxi_clean["rider_count"] == 0).sum(),
        (
            taxi_clean["dropoff_timestamp"] <
            taxi_clean["pickup_timestamp"]
        ).sum(),
        (taxi_clean["speed_mph"] > SPEED_THRESHOLD_MPH).sum(),
        0
    ]
})

display(before_after)

# 21. Post-Cleaning Validation

Re-run the intended rules after cleaning.

This demonstrates that the cleaning pipeline actually removed the observations it was designed to remove.

In [ ]:
# ============================================================
# 21. Post-cleaning validation
# ============================================================

checks = {
    "negative_fare_rows": int((taxi_clean["base_fare"] < 0).sum()),
    "zero_rider_rows": int((taxi_clean["rider_count"] == 0).sum()),
    "dropoff_before_pickup_rows": int(
        (
            taxi_clean["dropoff_timestamp"] <
            taxi_clean["pickup_timestamp"]
        ).sum()
    ),
    "unrealistic_speed_rows": int(
        (taxi_clean["speed_mph"] > SPEED_THRESHOLD_MPH).sum()
    )
}

validation_df = pd.DataFrame(
    checks.items(),
    columns=["check", "remaining_rows"]
)

display(validation_df)

assert all(validation_df["remaining_rows"] == 0)

print("PASS: all globally/main-cleaned anomaly checks are zero.")

# 22. Load the Zone Dataset

The Zone Dataset maps:

- `loc_id`
- `borough_name`
- `zone_name`
- `service_zone`

to the taxi location identifiers used by the main Taxi Dataset.

In [ ]:
# ============================================================
# 22. Locate and load Zone Dataset
# ============================================================

zone_files = sorted(RAW_ZONE_DIR.glob("*.csv"))

print("Zone files:", len(zone_files))

for file in zone_files:
    print("-", file.name)

if not zone_files:
    raise FileNotFoundError(
        "No Zone CSV found in data/raw/zones/."
    )

ZONE_FILE = zone_files[0]

zone_df = pd.read_csv(
    ZONE_FILE,
    low_memory=False
)

print("Zone shape:", zone_df.shape)
display(zone_df.head())

# 23. Validate Zone Dataset

`loc_id` should identify each taxi zone uniquely.

Check duplicates, missing IDs, and missing descriptive fields.

In [ ]:
# ============================================================
# 23. Zone quality checks
# ============================================================

required_zone_columns = [
    "loc_id",
    "borough_name",
    "zone_name",
    "service_zone"
]

missing_zone_columns = [
    c for c in required_zone_columns
    if c not in zone_df.columns
]

print("Missing required Zone columns:", missing_zone_columns)

print(
    "Duplicate loc_id rows:",
    zone_df["loc_id"].duplicated().sum()
)

print(
    "Missing loc_id rows:",
    zone_df["loc_id"].isna().sum()
)

display(
    zone_df[required_zone_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

# 24. Validate Taxi Location IDs

Check whether every pickup and destination location ID has a corresponding Zone Dataset record.

In [ ]:
# ============================================================
# 24. Location ID validation
# ============================================================

valid_zone_ids = set(
    zone_df["loc_id"].dropna()
)

invalid_origin = ~taxi_clean["origin_loc_id"].isin(valid_zone_ids)
invalid_destination = ~taxi_clean["dest_loc_id"].isin(valid_zone_ids)

invalid_origin_count = invalid_origin.sum()
invalid_destination_count = invalid_destination.sum()

print(
    f"Unmapped origin IDs: {invalid_origin_count:,} "
    f"({invalid_origin_count / len(taxi_clean) * 100:.4f}%)"
)

print(
    f"Unmapped destination IDs: {invalid_destination_count:,} "
    f"({invalid_destination_count / len(taxi_clean) * 100:.4f}%)"
)

# 25. Enrich Pickup and Drop-Off Locations

The Zone Dataset is joined twice:

1. `origin_loc_id` → origin zone information
2. `dest_loc_id` → destination zone information

The original location IDs are retained.

In [ ]:
# ============================================================
# 25. Zone enrichment
# ============================================================

pickup_zones = zone_df.rename(columns={
    "loc_id": "origin_loc_id",
    "borough_name": "origin_borough",
    "zone_name": "origin_zone",
    "service_zone": "origin_service_zone"
})

dropoff_zones = zone_df.rename(columns={
    "loc_id": "dest_loc_id",
    "borough_name": "dest_borough",
    "zone_name": "dest_zone",
    "service_zone": "dest_service_zone"
})

taxi_enriched = taxi_clean.merge(
    pickup_zones[
        [
            "origin_loc_id",
            "origin_borough",
            "origin_zone",
            "origin_service_zone"
        ]
    ],
    on="origin_loc_id",
    how="left",
    validate="many_to_one"
)

taxi_enriched = taxi_enriched.merge(
    dropoff_zones[
        [
            "dest_loc_id",
            "dest_borough",
            "dest_zone",
            "dest_service_zone"
        ]
    ],
    on="dest_loc_id",
    how="left",
    validate="many_to_one"
)

print("Enriched shape:", taxi_enriched.shape)

display(
    taxi_enriched[
        [
            "origin_loc_id",
            "origin_borough",
            "origin_zone",
            "dest_loc_id",
            "dest_borough",
            "dest_zone"
        ]
    ].head(10)
)

# 26. Validate Zone Join

In [ ]:
# ============================================================
# 26. Zone join validation
# ============================================================

zone_join_report = pd.DataFrame({
    "field": [
        "origin_zone",
        "origin_borough",
        "dest_zone",
        "dest_borough"
    ],
    "missing_count": [
        taxi_enriched["origin_zone"].isna().sum(),
        taxi_enriched["origin_borough"].isna().sum(),
        taxi_enriched["dest_zone"].isna().sum(),
        taxi_enriched["dest_borough"].isna().sum()
    ]
})

zone_join_report["missing_percentage"] = (
    zone_join_report["missing_count"] /
    len(taxi_enriched) *
    100
)

display(zone_join_report)

# 27. Create Basic Temporal Features

These are suitable starting features for later exploratory and predictive work.

They use pickup time only, so they do not directly reveal the future trip outcome.

In [ ]:
# ============================================================
# 27. Temporal features
# ============================================================

taxi_enriched["pickup_date"] = (
    taxi_enriched["pickup_timestamp"].dt.date
)

taxi_enriched["pickup_hour"] = (
    taxi_enriched["pickup_timestamp"].dt.hour
)

taxi_enriched["pickup_dayofweek"] = (
    taxi_enriched["pickup_timestamp"].dt.dayofweek
)

taxi_enriched["pickup_month"] = (
    taxi_enriched["pickup_timestamp"].dt.month
)

taxi_enriched["is_weekend"] = (
    taxi_enriched["pickup_dayofweek"] >= 5
)

display(
    taxi_enriched[
        [
            "pickup_timestamp",
            "pickup_hour",
            "pickup_dayofweek",
            "pickup_month",
            "is_weekend"
        ]
    ].head()
)

# 28. Fare-Prediction Leakage Check

The challenge defines fare prediction as a pre-trip task.

Therefore, variables that become known only after the trip should not be used as input features for the fare-prediction system.

Potential post-trip variables include:

- drop-off timestamp;
- actual trip duration;
- actual speed;
- driver tip;
- toll;
- final charge.

These fields may be useful for analysis, but they must not be treated as pre-trip predictors without a clearly justified availability assumption.

In [ ]:
# ============================================================
# 28. Potential leakage fields
# ============================================================

potential_post_trip = [
    "dropoff_timestamp",
    "trip_duration_seconds",
    "trip_duration_minutes",
    "trip_duration_hours",
    "speed_mph",
    "driver_tip_payment",
    "toll_total",
    "charge_total"
]

leakage_report = pd.DataFrame({
    "feature": [
        c for c in potential_post_trip
        if c in taxi_enriched.columns
    ],
    "safe_for_pre_trip_fare_prediction": False
})

display(leakage_report)

# 29. Save Final Outputs

Outputs:

1. quality-audit dataset;
2. cleaned Taxi dataset;
3. enriched Taxi dataset;
4. anomaly summary;
5. decision matrix;
6. monthly quality report;
7. final summary JSON.

The raw CSV files are never overwritten.

In [ ]:
# ============================================================
# 29. Save outputs
# ============================================================

clean_path = PROCESSED_DIR / "Urban_Flow_Analytics_Taxi_Clean_12Month.parquet"
enriched_path = PROCESSED_DIR / "Urban_Flow_Analytics_Taxi_Clean_Enriched_12Month.parquet"

summary_json_path = REPORT_DIR / "data_quality_summary.json"
anomaly_csv_path = REPORT_DIR / "anomaly_summary.csv"
decision_csv_path = REPORT_DIR / "data_quality_decision_matrix.csv"
monthly_csv_path = REPORT_DIR / "monthly_data_quality_summary.csv"

taxi_clean.to_parquet(clean_path, index=False)
taxi_enriched.to_parquet(enriched_path, index=False)

anomaly_summary.to_csv(anomaly_csv_path, index=False)
decision_matrix.to_csv(decision_csv_path, index=False)
monthly_quality.to_csv(monthly_csv_path)

final_summary = {
    "raw_rows": int(initial_rows),
    "clean_rows": int(len(taxi_clean)),
    "rows_removed": int(initial_rows - len(taxi_clean)),
    "rows_removed_percentage": float(
        (initial_rows - len(taxi_clean)) / initial_rows * 100
    ),
    "negative_fares": int(
        taxi_quality["flag_negative_fare"].sum()
    ),
    "negative_fares_percentage": float(
        taxi_quality["flag_negative_fare"].mean() * 100
    ),
    "zero_distance_nonzero_fare": int(
        taxi_quality["flag_zero_distance_nonzero_fare"].sum()
    ),
    "zero_distance_nonzero_fare_percentage": float(
        taxi_quality["flag_zero_distance_nonzero_fare"].mean() * 100
    ),
    "zero_riders": int(
        taxi_quality["flag_zero_riders"].sum()
    ),
    "zero_riders_percentage": float(
        taxi_quality["flag_zero_riders"].mean() * 100
    ),
    "dropoff_before_pickup": int(
        taxi_quality["flag_dropoff_before_pickup"].sum()
    ),
    "dropoff_before_pickup_percentage": float(
        taxi_quality["flag_dropoff_before_pickup"].mean() * 100
    ),
    "unrealistic_speed": int(
        taxi_quality["flag_unrealistic_speed"].sum()
    ),
    "unrealistic_speed_percentage": float(
        taxi_quality["flag_unrealistic_speed"].mean() * 100
    ),
    "unique_rows_with_any_required_anomaly": int(
        taxi_quality["any_required_anomaly"].sum()
    ),
    "unique_rows_with_any_required_anomaly_percentage": float(
        taxi_quality["any_required_anomaly"].mean() * 100
    ),
    "speed_threshold_mph": float(SPEED_THRESHOLD_MPH)
}

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("Saved:")
for path in [
    quality_audit_path,
    clean_path,
    enriched_path,
    summary_json_path,
    anomaly_csv_path,
    decision_csv_path,
    monthly_csv_path
]:
    print("-", path)

# 30. Final Reproducibility Checks

In [ ]:
# ============================================================
# 30. Final checks
# ============================================================

assert len(taxi_files) == 12
assert initial_rows > 0
assert len(taxi_clean) > 0
assert len(zone_df) > 0

for flag in FLAG_COLUMNS:
    assert flag in taxi_quality.columns

for path in [
    quality_audit_path,
    clean_path,
    enriched_path,
    summary_json_path
]:
    assert path.exists()

print("=" * 70)
print("FINAL DATA QUALITY PIPELINE")
print("=" * 70)
print(f"Taxi monthly files loaded : {len(taxi_files)}")
print(f"Raw rows                  : {initial_rows:,}")
print(f"Clean rows                : {len(taxi_clean):,}")
print(f"Enriched rows             : {len(taxi_enriched):,}")
print(f"Zone rows                 : {len(zone_df):,}")
print(f"Speed threshold           : {SPEED_THRESHOLD_MPH:.1f} mph")
print("Required anomaly flags    : PASS")
print("Cleaning validation       : PASS")
print("Output files              : PASS")
print("=" * 70)

# 31. Final Findings Template

After executing the notebook on the real 12-month dataset, replace the placeholders below with the values produced by the notebook.

## Negative fares

- Rows affected: **calculated value**
- Percentage: **calculated percentage**
- Treatment: exclude from normal fare-prediction data.
- Justification: negative fare values are not safely recoverable by imputation.

## Zero distance + non-zero fare

- Rows affected: **calculated value**
- Percentage: **calculated percentage**
- Treatment: retain globally; filter from movement/speed analysis.
- Justification: zero measured distance does not necessarily prove the fare is invalid.

## Zero rider count

- Rows affected: **calculated value**
- Percentage: **calculated percentage**
- Treatment: filter from analyses requiring a valid passenger count.
- Justification: the actual rider count cannot be reliably inferred.

## Drop-off before pickup

- Rows affected: **calculated value**
- Percentage: **calculated percentage**
- Treatment: drop.
- Justification: the temporal ordering is impossible and the correct timestamp cannot be inferred.

## Unrealistic speed

- Rows affected: **calculated value**
- Percentage: **calculated percentage**
- Threshold: **calculated/documented threshold**
- Treatment: filter from movement/travel-time analyses.
- Justification: the speed is operationally implausible and should not be artificially imputed.

## Overall

- Raw rows: **calculated value**
- Clean rows: **calculated value**
- Main cleaning rows removed: **calculated value**
- Main cleaning percentage: **calculated percentage**
- Unique rows with at least one required anomaly: **calculated value**
- Unique anomaly percentage: **calculated percentage**

# 32. Conclusion

The complete 12-month Urban Flow Analytics Taxi Dataset was assessed before predictive analysis.

The workflow validates the monthly schemas, preserves the raw data, analyzes missing values and duplicates, derives duration and speed, identifies the five required anomaly categories, quantifies their impact, documents treatment decisions, investigates monthly variation, validates taxi location IDs against the Zone Dataset, and creates a reproducible cleaned/enriched dataset.

The cleaning strategy avoids unsupported imputation. Impossible temporal observations are removed, while anomalies that do not necessarily prove a record is globally invalid are handled with task-specific filters.

The resulting dataset is ready for the next stages of the Datathon:

- feature engineering;
- train/validation/test preparation;
- fare prediction;
- trip-time prediction;
- demand forecasting;
- spatial-temporal analysis.

# Appendix — Large Dataset Processing

If the complete 12-month dataset cannot fit comfortably in memory, process each CSV in chunks.

The key principle remains the same: apply identical validation/cleaning logic to every chunk and preserve a reproducible audit trail.

In [ ]:
# ============================================================
# Chunk-processing pattern for very large CSV files
# ============================================================

def taxi_chunks(file_path, chunksize=500_000):
    for chunk in pd.read_csv(
        file_path,
        chunksize=chunksize,
        low_memory=False
    ):
        chunk["source_file"] = Path(file_path).name
        yield chunk

# Example:
#
# for chunk in taxi_chunks(taxi_files[0]):
#     print(chunk.shape)
#     # Standardize types
#     # Create anomaly flags
#     # Write processed chunk to Parquet

# Appendix — Submission Checklist

- [ ] All 12 monthly Taxi files loaded.
- [ ] February sample not used as the final dataset by itself.
- [ ] Zone Dataset loaded.
- [ ] Raw data preserved.
- [ ] Missing values analyzed.
- [ ] Duplicate records analyzed.
- [ ] Negative fares quantified.
- [ ] Zero-distance/non-zero-fare trips quantified.
- [ ] Zero rider counts quantified.
- [ ] Drop-off-before-pickup trips quantified.
- [ ] Unrealistic speeds quantified.
- [ ] Percentage of total raw rows reported for every anomaly.
- [ ] Treatment and justification documented for every anomaly.
- [ ] Anomaly overlap analyzed.
- [ ] Monthly anomaly rates analyzed.
- [ ] Zone IDs validated.
- [ ] Zone enrichment validated.
- [ ] Post-cleaning checks passed.
- [ ] Potential prediction leakage documented.
- [ ] Clean dataset exported.
- [ ] Quality audit exported.
- [ ] Notebook is reproducible from raw data.

## 28b. Categorical Label Encoding (Model-Ready Dataset)

Two encoding strategies are produced:

| Strategy | Use case |
|---|---|
| **Ordinal / integer codes** | Tree-based models (XGBoost, LightGBM, RandomForest) |
| **One-hot encoding** | Linear models, neural networks |

The raw string columns are retained in `taxi_enriched`.
The encoded versions are stored in a separate `taxi_model_ready` DataFrame
so the original human-readable values are not lost.

> **Note**: Zone/borough columns (`origin_borough`, `dest_borough`,
> `origin_service_zone`, `dest_service_zone`) have been joined from the
> Zone Dataset and are also encoded here.

In [ ]:
# ============================================================
# 28b. Categorical encoding
# ============================================================

# All categorical columns to encode (raw + zone-joined)
ENCODE_COLS = [
    col for col in CATEGORY_COLS
    + [
        "origin_borough",
        "dest_borough",
        "origin_service_zone",
        "dest_service_zone",
    ]
    if col in taxi_enriched.columns
]

# ---------------------------------------------------------------
# Strategy 1 — Ordinal integer codes  (cat.codes)
# -1 = NaN / unknown
# ---------------------------------------------------------------
taxi_model_ready = taxi_enriched.copy()

label_maps = {}   # store label → int mappings for documentation

for col in ENCODE_COLS:
    taxi_model_ready[col] = taxi_model_ready[col].astype("category")
    # Save the mapping before encoding
    label_maps[col] = dict(
        enumerate(taxi_model_ready[col].cat.categories)
    )
    taxi_model_ready[col + "_code"] = taxi_model_ready[col].cat.codes

print("=== Ordinal Encoding Maps ===")
for col, mapping in label_maps.items():
    print(f"\n{col}:")
    for code, label in mapping.items():
        print(f"  {code:3d} => {label}")

# ---------------------------------------------------------------
# Strategy 2 — One-hot encoding  (get_dummies)
# Only low-cardinality columns (≤ 20 unique values)
# ---------------------------------------------------------------
low_cardinality_cols = [
    col for col in ENCODE_COLS
    if taxi_enriched[col].nunique(dropna=True) <= 20
]

taxi_ohe = pd.get_dummies(
    taxi_enriched[low_cardinality_cols + ["base_fare"]],
    columns=low_cardinality_cols,
    drop_first=False,   # keep all dummies; drop_first in model step
    dtype="uint8"
)

print(f"\n=== One-Hot Encoded Columns ===")
print(f"Input categorical columns  : {low_cardinality_cols}")
print(f"Output columns after OHE   : {taxi_ohe.shape[1]}")
display(taxi_ohe.head())

# ---------------------------------------------------------------
# Encoding quality report
# ---------------------------------------------------------------
encoding_report = pd.DataFrame([
    {
        "column": col,
        "n_unique": taxi_enriched[col].nunique(dropna=True),
        "null_count": taxi_enriched[col].isna().sum(),
        "strategy": "ordinal" if col not in low_cardinality_cols else "ordinal + one-hot",
        "ordinal_col": col + "_code",
        "ohe_cols": (
            [c for c in taxi_ohe.columns if c.startswith(col + "_")]
            if col in low_cardinality_cols else []
        )
    }
    for col in ENCODE_COLS
])

display(encoding_report)

print(f"\ntaxi_model_ready shape : {taxi_model_ready.shape}")
print(f"taxi_ohe shape         : {taxi_ohe.shape}")


## 28c. Save Encoding Maps

The label-to-integer mapping is saved as JSON so it can be reused
in the modelling notebook without re-fitting on training data.

In [ ]:
# ============================================================
# 28c. Save encoding maps + model-ready parquet
# ============================================================

import json as _json

label_map_path = REPORT_DIR / "categorical_label_maps.json"

# Convert int keys to str for JSON serialisability
label_maps_serialisable = {
    col: {str(k): v for k, v in mapping.items()}
    for col, mapping in label_maps.items()
}

with open(label_map_path, "w", encoding="utf-8") as _f:
    _json.dump(label_maps_serialisable, _f, indent=2, ensure_ascii=False)

model_ready_path = PROCESSED_DIR / "Urban_Flow_Analytics_Taxi_Model_Ready_12Month.parquet"
taxi_model_ready.to_parquet(model_ready_path, index=False)

print(f"Label maps saved   : {label_map_path}")
print(f"Model-ready saved  : {model_ready_path}")
